In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List

In [2]:

# Capture exactly 6 comma-separated fields, allowing optional spaces.
SIX_TUPLE_RE = re.compile(
    r"\(\s*([^,()]+?)\s*,\s*([^,()]+?)\s*,\s*([^,()]+?)\s*,\s*([^,()]+?)\s*,\s*([^,()]+?)\s*,\s*([^,()]+?)\s*\)"
)


def replace_six_tuples(text: str, repl: Dict[str, str]) -> str:
    """Replace selected values in each matched 6-item tuple.

    repl keys are existing field values, and repl values are replacement strings.
    """

    def _sub(match: re.Match[str]) -> str:
        items: List[str] = [match.group(i) for i in range(1, 7)]
        for i, value in enumerate(items):
            if value in repl:
                items[i] = repl[value]
        return f"({','.join(items)})"

    return SIX_TUPLE_RE.sub(_sub, text)


def parse_map(entries: List[str]) -> Dict[str, str]:
    """Parse CLI mappings of the form old=new old2=new2 ..."""
    out: Dict[str, str] = {}
    for item in entries:
        if "=" not in item:
            raise ValueError(f"Invalid mapping '{item}'. Use format old=new")
        old, new = item.split("=", 1)
        old = old.strip()
        new = new.strip()
        if not old:
            raise ValueError(f"Invalid mapping '{item}'. Left side cannot be empty.")
        out[old] = new
    return out

In [3]:
# File created by Mathematica notebook
fname = './lffvvnpBox.txt'
# fname = './lffvvnpTri.txt'
# fname = './lffvvnpSelf.txt'
with open(fname,'r') as f:
    data = f.read()
data = data.replace('"',"'")

In [4]:
# Break into color tensor blocks:
blocks = {x.strip().split('\n')[0] : x.strip().split('\n')[1:] for x in data.split("'------------------'\n")[1:]}

In [5]:
for tblock in blocks:
    lffvvList = blocks[tblock]
    lffvvDict = dict(zip(lffvvList[0:-1:2],lffvvList[1::2]))
    blocks[tblock] = lffvvDict

In [6]:
replacements = {'t' : '( P(-1,1)**2 + P(-1,3)**2 + 2*P(-1,1)*P(-1,3) )',
                'u' : '( P(-1,2)**2 + P(-1,3)**2 + 2*P(-1,2)*P(-1,3) )',
                's' : '( P(-1,1)**2 + P(-1,2)**2 + 2*P(-1,1)*P(-1,2) )',
                'p1sq' : 'P(-1,1)**2',
                'p2sq' : 'P(-1,2)**2'
               }

In [7]:
lffvvBlock = "%s = Lorentz(name = %s,\n\
                spins = [ 2, 2, 3, 3 ],\n\
                structure = '%s')"

### The output below should go into lorentz.py

In [8]:
for t,lffvvDict in blocks.items():
    for ffvvname,ffvvexpr in lffvvDict.items():
        expr = ffvvexpr[:]
        newExpr = replace_six_tuples(expr, replacements)
        exprLines = eval(newExpr).split('\\\n')
        exprNew = ('\\\n').ljust(33).join(exprLines)
        print(lffvvBlock %(ffvvname.replace("'",''),ffvvname,exprNew))
        print('\n')


FFVVNP1 = Lorentz(name = 'FFVVNP1',
                spins = [ 2, 2, 3, 3 ],
                structure = '( Gamma(3,2,-1)*ProjP(-1,1) )*(\
                                 P(4,1)*(2*(2*D001c(0,P(-1,1)**2,P(-1,2)**2,0,( P(-1,2)**2 + P(-1,3)**2 + 2*P(-1,2)*P(-1,3) ),( P(-1,1)**2 + P(-1,2)**2 + 2*P(-1,1)*P(-1,2) )) + D00d(0,0,P(-1,1)**2,P(-1,2)**2,( P(-1,1)**2 + P(-1,2)**2 + 2*P(-1,1)*P(-1,2) ),( P(-1,2)**2 + P(-1,3)**2 + 2*P(-1,2)*P(-1,3) ))))\
                                  + P(4,2)*(2*(2*(D001b(P(-1,1)**2,P(-1,2)**2,0,0,( P(-1,1)**2 + P(-1,2)**2 + 2*P(-1,1)*P(-1,2) ),( P(-1,2)**2 + P(-1,3)**2 + 2*P(-1,2)*P(-1,3) )) + D001c(0,P(-1,1)**2,P(-1,2)**2,0,( P(-1,2)**2 + P(-1,3)**2 + 2*P(-1,2)*P(-1,3) ),( P(-1,1)**2 + P(-1,2)**2 + 2*P(-1,1)*P(-1,2) ))) + D00d(0,0,P(-1,1)**2,P(-1,2)**2,( P(-1,1)**2 + P(-1,2)**2 + 2*P(-1,1)*P(-1,2) ),( P(-1,2)**2 + P(-1,3)**2 + 2*P(-1,2)*P(-1,3) ))))\
                                  + P(4,3)*(-2*(2*D001d(0,0,P(-1,1)**2,P(-1,2)**2,( P(-1,1)**2 + P(-1,2)**2 + 

### For cross-checking the results:

In [9]:
for t,lffvvDict in blocks.items():
    for ffvvname,ffvvexpr in lffvvDict.items():
        expr = ffvvexpr[:]
        exprLines = eval(expr).split('\\\n')
        exprNew = ('\\\n').ljust(33).join(exprLines)
        print(lffvvBlock %(ffvvname.replace("'",''),ffvvname,exprNew))
        print('\n')


FFVVNP1 = Lorentz(name = 'FFVVNP1',
                spins = [ 2, 2, 3, 3 ],
                structure = '( Gamma(3,2,-1)*ProjP(-1,1) )*(\
                                 P(4,1)*(2*(2*D001c(0,p1sq,p2sq,0,u,s) + D00d(0,0,p1sq,p2sq,s,u)))\
                                  + P(4,2)*(2*(2*(D001b(p1sq,p2sq,0,0,s,u) + D001c(0,p1sq,p2sq,0,u,s)) + D00d(0,0,p1sq,p2sq,s,u)))\
                                  + P(4,3)*(-2*(2*D001d(0,0,p1sq,p2sq,s,u) + D00d(0,0,p1sq,p2sq,s,u)))\
                                 )')


FFVVNP2 = Lorentz(name = 'FFVVNP2',
                spins = [ 2, 2, 3, 3 ],
                structure = '( Gamma(4,2,-1)*ProjP(-1,1) )*(\
                                 P(3,1)*(4*D001c(0,p1sq,p2sq,0,u,s))\
                                  + P(3,2)*(4*(D001b(p1sq,p2sq,0,0,s,u) + D001c(0,p1sq,p2sq,0,u,s)))\
                                  + P(3,3)*(-2*(2*D001d(0,0,p1sq,p2sq,s,u) + D00d(0,0,p1sq,p2sq,s,u)))\
                                 )')


FFVVNP3 = Lorentz(name = 'FFVVNP3'